In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.tools import StructuredTool, InjectedToolArg
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
from typing import Annotated
from dotenv import load_dotenv
import requests
import json
import os

c:\Users\thaku\Desktop\LANGCHAIN_TUTORIAL\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

True

In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [4]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [5]:
multiply.name

'multiply'

In [6]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [8]:
# Defining Model

llm = HuggingFaceEndpoint(
    repo_id="MiniMaxAI/MiniMax-M2.5",
    task="text-generation"
)

model_1 = ChatHuggingFace(llm=llm)

In [9]:
# tool binding

llm_with_tools = model_1.bind_tools([multiply])

In [10]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hi there! I'm doing well, thank you for asking! 😊\n\nI'm ready to help you with whatever you need. Whether it's calculations, answering questions, or working on tasks together, I'm here for you.\n\nHow can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 211, 'total_tokens': 294}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c9a69-fb01-7fa1-b716-2448779fad96-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 211, 'output_tokens': 83, 'total_tokens': 294})

In [11]:
llm_with_tools.invoke("Can you multiply 2 with 10")

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 2, "b": 10}', 'name': 'multiply', 'description': None}, 'id': 'call_function_90oa2yedwq65_1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 215, 'total_tokens': 346}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9a6a-1519-7531-bd75-dc96af7d7bfd-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 10}, 'id': 'call_function_90oa2yedwq65_1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 215, 'output_tokens': 131, 'total_tokens': 346})

In [12]:
llm_with_tools.invoke("Can you multiply 2 with 10").tool_calls[0]

{'name': 'multiply',
 'args': {'a': 2, 'b': 10},
 'id': 'call_function_zx1u9l2ei56h_1',
 'type': 'tool_call'}

In [13]:
query = HumanMessage('can you multiply 7 with 1000')

In [14]:
messages = [query]

In [15]:
messages

[HumanMessage(content='can you multiply 7 with 1000', additional_kwargs={}, response_metadata={})]

In [16]:
result = llm_with_tools.invoke(messages)

In [17]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 7, 'b': 1000},
 'id': 'call_function_qsbxa9wr8zbt_1',
 'type': 'tool_call'}

In [18]:
messages.append(result)

In [19]:
messages

[HumanMessage(content='can you multiply 7 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 7, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_function_qsbxa9wr8zbt_1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 216, 'total_tokens': 283}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9a6a-4d41-7601-8f88-525fb118871d-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 1000}, 'id': 'call_function_qsbxa9wr8zbt_1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 67, 'total_tokens': 283})]

In [20]:
tool_result = multiply.invoke(result.tool_calls[0])

In [21]:
tool_result

ToolMessage(content='7000', name='multiply', tool_call_id='call_function_qsbxa9wr8zbt_1')

In [22]:
messages.append(tool_result)

In [23]:
messages

[HumanMessage(content='can you multiply 7 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 7, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_function_qsbxa9wr8zbt_1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 216, 'total_tokens': 283}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9a6a-4d41-7601-8f88-525fb118871d-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 1000}, 'id': 'call_function_qsbxa9wr8zbt_1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 67, 'total_tokens': 283}),
 ToolMessage(content='7000', name='multiply', tool_call_id='call_function_qsbxa9wr8zbt_1')]

In [24]:
llm_with_tools.invoke(messages).content

'7 multiplied by 1000 equals **7000**.'

In [51]:
# GETTING API KEY
API_KEY = os.getenv("EXCHANGERATE_API_KEY")

# tool create
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/pair/{base_currency}/{target_currency}"

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [52]:
class CurrencyToolkit:
    def get_tools(self):
        return [
            get_conversion_factor,
            convert
        ]

In [53]:
toolkit = CurrencyToolkit()
tools = toolkit.get_tools()

In [54]:
tools

[StructuredTool(name='get_conversion_factor', description='This function fetches the currency conversion factor between a given base currency and a target currency', args_schema=<class 'langchain_core.utils.pydantic.get_conversion_factor'>, func=<function get_conversion_factor at 0x0000020449A4A0C0>),
 StructuredTool(name='convert', description='given a currency conversion rate this function calculates the target currency value from a given base currency value', args_schema=<class 'langchain_core.utils.pydantic.convert'>, func=<function convert at 0x000002044EB28AE0>)]

In [55]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [56]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1772064002,
 'time_last_update_utc': 'Thu, 26 Feb 2026 00:00:02 +0000',
 'time_next_update_unix': 1772150402,
 'time_next_update_utc': 'Fri, 27 Feb 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.9653}

In [57]:
convert.invoke({'base_currency_value':10, 'conversion_rate':90.9645})

909.645

In [58]:
model_2 = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [59]:
# tool binding

llm_with_tools = model_2.bind_tools(tools)

In [60]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that convert 10 USD to INR')]

In [61]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [62]:
ai_message = llm_with_tools.invoke(messages)

In [39]:
ai_message

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'fc738430-757a-4fd7-8a91-f450919f18ad': 'EpwICpkIAb4+9vubwkgvAkDbTxz12G3iOIKmBUw4jOZ1tRHRuXupSlP0C34if1JZfqzCykSHHkVMDgc1pO5FKCx8p3B+5VnRmz4p17pfuxLC9UXmojHd8H0ydPTD86jcqc3O/o6nUxx6ldPSD23OG8zeISdYn6yUj4Tf8Tw940aReL1swbgyoL2Dtggr7+cN5hlSex1H5WphGeK4BXj5CLfc3OSgOVsaOghS/RZ4U49DXNR5MvcyTt+fikd67nHhYC7qpmOO5w0n5ABbB3DZwLY6Dk0KTL85GuIVlsZQ9StoXnAAKpT1wiJMNUPb3UgoLRz+lXLidSec7/0202wLb2EVbhKjhoGrzxQnYV67HQdQI1IgFgjFbXRIG+GqCWbEJp5OQdIvNwBUNi0qRDwK9YiMKauoZB/JfMw/EETRbA6qH+q/B6S7ysqxoKc8W+wOS99Fk3t3/AsNfHi0fFKptzqVfU8s7tzIIEdFz9c0wU6EoHuOAhipyFpC+fTz//80+7WXo4rbL611skNe4brytyEvDbAQPltcJ6zRNS2AE8knvpxnXta4ncuiidU1d+W5ULBDoHH93v4Z6NiQaCkEIVQquiMQswX7TRPQB4cGuOEEafLHG6H7PsEYK1dKvAXq7eVKQZ63GL7OOMPh6rXlr3P8ajCqkSYApW5jsVyb7AQStmKnI21AlDFc+I3xTeCRJNCQg+FTT6PYvpm9SXG09cCjpAhyuWr6uurTv52EtgK5aXw

In [40]:
messages.append(ai_message)

In [41]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'fc738430-757a-4fd7-8a91-f450919f18ad': 'EpwICpkIAb4+9vubwkgvAkDbTxz12G3iOIKmBUw4jOZ1tRHRuXupSlP0C34if1JZfqzCykSHHkVMDgc1pO5FKCx8p3B+5VnRmz4p17pfuxLC9UXmojHd8H0ydPTD86jcqc3O/o6nUxx6ldPSD23OG8zeISdYn6yUj4Tf8Tw940aReL1swbgyoL2Dtggr7+cN5hlSex1H5WphGeK4BXj5CLfc3OSgOVsaOghS/RZ4U49DXNR5MvcyTt+fikd67nHhYC7qpmOO5w0n5ABbB3DZwLY6Dk0KTL85GuIVlsZQ9StoXnAAKpT1wiJMNUPb3UgoLRz+lXLidSec7/0202wLb2EVbhKjhoGrzxQnYV67HQdQI1IgFgjFbXRIG+GqCWbEJp5OQdIvNwBUNi0qRDwK9YiMKauoZB/JfMw/EETRbA6qH+q/B6S7ysqxoKc8W+wOS99Fk3t3/AsNfHi0fFKptzqVfU8s7tzIIEdFz9c0wU6EoHuOAhipyFpC+fTz//80+7WXo4rbL611skNe4brytyEvDbAQPltcJ6zRNS2AE8knvpxnXta4ncuiidU1d+W5ULBDo

In [63]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '43ed176a-7979-4b9a-a29f-f58c6857b29d',
  'type': 'tool_call'}]

In [65]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)


In [66]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1772064002, "time_last_update_utc": "Thu, 26 Feb 2026 00:00:02 +0000", "time_next_update_unix": 1772150402, "time_next_update_utc": "Fri, 27 Feb 2026 00:00:02 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 90.9653}', name='get_conversion_factor', tool_call_id='43ed176a-7979-4b9a-a29f-f58c6857b29d')]

In [67]:
llm_with_tools.invoke(messages).content

''